# LAB02 · Búsqueda, selección y auditoría de un dataset público

**Dataset:** German Credit Data (UCI Machine Learning Repository, id=144)

**Dominio:** Riesgo crediticio (finanzas)

**Unidad de análisis:** Cada fila representa una solicitud de crédito individual.

**Target:** Riesgo crediticio (buen riesgo / mal riesgo).

**Decisión que apoya:** Determinar si un banco debería aprobar o rechazar una solicitud de crédito.

**Error más costoso:** Clasificar como "buen riesgo" a alguien que en realidad es "mal riesgo" (falso negativo) — el banco pierde dinero al otorgar un crédito que no se pagará.

In [1]:
import sys
sys.path.append("../src")

from inf8239_u01.data import download_german_credit

path = download_german_credit()
print(path)

data\raw\dataset.csv


In [4]:
from inf8239_u01.data import download_german_credit

path = download_german_credit(destination="../data/raw/dataset.csv")
print(path)

..\data\raw\dataset.csv


In [3]:
import os
for root, dirs, files in os.walk(".."):
    if "dataset.csv" in files:
        print(os.path.join(root, "dataset.csv"))

..\notebooks\data\raw\dataset.csv


In [5]:
import pandas as pd
df = pd.read_csv("../data/raw/dataset.csv")
print(df.shape)
print(df.dtypes)
print(df.head())

(1000, 21)
Attribute1       str
Attribute2     int64
Attribute3       str
Attribute4       str
Attribute5     int64
Attribute6       str
Attribute7       str
Attribute8     int64
Attribute9       str
Attribute10      str
Attribute11    int64
Attribute12      str
Attribute13    int64
Attribute14      str
Attribute15      str
Attribute16    int64
Attribute17      str
Attribute18    int64
Attribute19      str
Attribute20      str
class          int64
dtype: object
  Attribute1  Attribute2 Attribute3 Attribute4  Attribute5 Attribute6  \
0        A11           6        A34        A43        1169        A65   
1        A12          48        A32        A43        5951        A61   
2        A14          12        A34        A46        2096        A61   
3        A11          42        A32        A42        7882        A61   
4        A11          24        A33        A40        4870        A61   

  Attribute7  Attribute8 Attribute9 Attribute10  ...  Attribute12 Attribute13  \
0        A75  

In [6]:
print(df["class"].value_counts())

class
1    700
2    300
Name: count, dtype: int64


In [7]:
audit = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "ausentes": df.isna().sum(),
    "porcentaje_ausente": (df.isna().mean()*100).round(2),
    "unicos": df.nunique(dropna=False)
}).sort_values("porcentaje_ausente", ascending=False)
print("Duplicados:", df.duplicated().sum())
display(audit)

Duplicados: 0


,tipo,ausentes,porcentaje_ausente,unicos
Attribute1,str,0,0.0,4
Attribute12,str,0,0.0,4
Attribute20,str,0,0.0,2
Attribute19,str,0,0.0,2
Attribute18,int64,0,0.0,2
Attribute17,str,0,0.0,4
Attribute16,int64,0,0.0,4
Attribute15,str,0,0.0,3
Attribute14,str,0,0.0,3
Attribute13,int64,0,0.0,53


In [8]:
TARGET = "class"
DROP_COLUMNS = []  # no hay columnas de ID ni fuga evidente en este dataset

X = df.drop(columns=[TARGET] + DROP_COLUMNS)
y = df[TARGET]
print(y.value_counts(dropna=False))
assert y.notna().all()
assert y.nunique() >= 2

class
1    700
2    300
Name: count, dtype: int64


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler())])
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                     ("onehot", OneHotEncoder(handle_unknown="ignore"))])

preprocess = ColumnTransformer([("num", num_pipe, num_cols),
                                ("cat", cat_pipe, cat_cols)])
print(len(num_cols), len(cat_cols))
print(num_cols)
print(cat_cols)

7 13
['Attribute2', 'Attribute5', 'Attribute8', 'Attribute11', 'Attribute13', 'Attribute16', 'Attribute18']
['Attribute1', 'Attribute3', 'Attribute4', 'Attribute6', 'Attribute7', 'Attribute9', 'Attribute10', 'Attribute12', 'Attribute14', 'Attribute15', 'Attribute17', 'Attribute19', 'Attribute20']


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, f1_score

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

dummy = Pipeline([("prep", preprocess), ("model", DummyClassifier(strategy="most_frequent"))])
svm = Pipeline([("prep", preprocess), ("model", SVC(C=1, gamma="scale", probability=True, random_state=42))])

for name, model in {"dummy": dummy, "svm": svm}.items():
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    print(name, f1_score(yte, pred, average="macro"))

print(classification_report(yte, svm.predict(Xte)))

dummy 0.4117647058823529


c:\Users\PC\INF8239_U01\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


svm 0.72
              precision    recall  f1-score   support

           1       0.81      0.92      0.86       140
           2       0.72      0.48      0.58        60

    accuracy                           0.79       200
   macro avg       0.77      0.70      0.72       200
weighted avg       0.78      0.79      0.78       200



In [11]:
svm_balanced = Pipeline([("prep", preprocess),
                          ("model", SVC(C=1, gamma="scale", class_weight="balanced",
                                        probability=True, random_state=42))])
svm_balanced.fit(Xtr, ytr)
pred_balanced = svm_balanced.predict(Xte)
print("F1 macro (balanced):", f1_score(yte, pred_balanced, average="macro"))
print(classification_report(yte, pred_balanced))

c:\Users\PC\INF8239_U01\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


F1 macro (balanced): 0.7362637362637363
              precision    recall  f1-score   support

           1       0.88      0.76      0.82       140
           2       0.57      0.77      0.66        60

    accuracy                           0.76       200
   macro avg       0.73      0.76      0.74       200
weighted avg       0.79      0.76      0.77       200



In [12]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = GridSearchCV(
    estimator=svm,
    param_grid={
        "model__C": [0.1, 1, 10],
        "model__gamma": ["scale", 0.01, 0.1],
        "model__class_weight": [None, "balanced"]
    },
    scoring="f1_macro", cv=cv, n_jobs=-1, return_train_score=True
)
search.fit(Xtr, ytr)
print(search.best_params_)
print("CV:", round(search.best_score_, 4))

c:\Users\PC\INF8239_U01\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


{'model__C': 1, 'model__class_weight': 'balanced', 'model__gamma': 'scale'}
CV: 0.6927


In [13]:
cv_results = pd.DataFrame(search.cv_results_)
columns = ["param_model__C", "param_model__gamma", "param_model__class_weight",
           "mean_test_score", "std_test_score", "mean_fit_time"]
display(cv_results[columns].sort_values("mean_test_score", ascending=False).head(10))

best = search.best_estimator_
best_pred = best.predict(Xte)
print(classification_report(yte, best_pred, digits=4))

,param_model__C,param_model__gamma,param_model__class_weight,mean_test_score,std_test_score,mean_fit_time
9,1.0,scale,balanced,0.692700,0.027906,0.896650
16,10.0,0.01,balanced,0.685880,0.033618,0.837907
11,1.0,0.1,balanced,0.685833,0.030955,1.022131
10,1.0,0.01,balanced,0.675984,0.034025,0.903016
3,0.1,scale,balanced,0.669197,0.025499,1.477476
14,10.0,0.1,NaN,0.668788,0.049258,1.177652
12,10.0,scale,NaN,0.667951,0.039911,1.380620
15,10.0,scale,balanced,0.666247,0.037550,1.143275
4,0.1,0.01,balanced,0.664707,0.041616,1.285823
17,10.0,0.1,balanced,0.661541,0.039934,0.932379


              precision    recall  f1-score   support

           1     0.8833    0.7571    0.8154       140
           2     0.5750    0.7667    0.6571        60

    accuracy                         0.7600       200
   macro avg     0.7292    0.7619    0.7363       200
weighted avg     0.7908    0.7600    0.7679       200



In [14]:
from pathlib import Path
import joblib

Path("../reports").mkdir(exist_ok=True)
cv_results[columns].to_csv("../reports/credit_cv_results.csv", index=False)
joblib.dump(best, "../reports/credit_best.joblib")

['../reports/credit_best.joblib']

Conclusión — Ejercicio 01

Este ejercicio permitió construir un pipeline de clasificación reproducible y validar, en dos etapas, los principios de diseño experimental discutidos en la Unidad 01. En primer lugar, se replicó el procedimiento guiado sobre el dataset de práctica Breast Cancer, donde una SVM con kernel RBF alcanzó un F1 macro de 0.981 frente a un baseline de 0.387, confirmando que el escalado dentro de un Pipeline y la validación cruzada estratificada evitan la fuga de datos y producen una estimación confiable del desempeño.

La parte central del ejercicio consistió en aplicar el mismo procedimiento a un dataset real de riesgo crediticio (German Credit, UCI). La auditoría inicial no encontró duplicados ni valores ausentes, y se confirmó que las 20 variables predictoras están disponibles en el momento de la solicitud del crédito, por lo que no existe riesgo de fuga temporal. El principal hallazgo metodológico surgió al comparar una SVM estándar contra una versión con class_weight="balanced": la primera alcanzó un F1 macro de 0.72 pero con un recall de apenas 0.48 en la clase "mal riesgo"; la versión balanceada elevó ese recall a 0.77, a costa de una ligera caída en precisión y accuracy general.

Esta decisión no fue arbitraria: el propio dataset incluye una matriz de costos que establece que un falso negativo (aprobar un crédito a un mal pagador) es cinco veces más costoso que un falso positivo (rechazar a un buen pagador). Bajo ese criterio de negocio, sacrificar accuracy general a cambio de duplicar la detección de malos riesgos es una decisión defendible, no simplemente una mejora numérica aislada. La búsqueda sistemática con GridSearchCV (18 combinaciones de C, gamma y class_weight) confirmó esta intuición de forma objetiva: nueve de las diez mejores configuraciones según validación cruzada utilizaban balanceo de clases.

Entre las limitaciones del trabajo destaca el tamaño modesto del dataset (1,000 filas), que produce desviaciones estándar de hasta 0.04 entre pliegues de validación cruzada, y la antigüedad de las variables (codificadas en Marcos Alemanes de los años 90), lo que limita su vigencia para un contexto crediticio actual. Tampoco se realizó calibración de probabilidades, por lo que el umbral de decisión implícito (0.5) no fue optimizado explícitamente para el problema de negocio.

Como siguiente paso, sería valioso explorar CalibratedClassifierCV para obtener probabilidades confiables y ajustar el umbral de decisión directamente sobre la matriz de costos del banco, en lugar de depender únicamente del balanceo de clases durante el entrenamiento. Este análisis se retomará en el Ejercicio 02, donde se compararán ensambles adicionales bajo el mismo protocolo.